<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/10-protection-security-and-isolation.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Protection, Security, and Isolation**

The previous chapter followed a pathname to a dentry and inode, mapped file offsets to storage, and established when updates become durable. One question was intentionally postponed: **after the kernel has identified an object, why is this process allowed to read, modify, execute, signal, map, mount, or delegate it?**

That question begins with protection but does not end there:

- **protection** defines and enforces which operations subjects may perform on objects;
- **isolation** limits what one execution domain can observe, affect, or consume outside itself;
- **security** asks whether the complete system preserves its intended properties despite adversarial inputs, compromised components, races, resource exhaustion, misconfiguration, and failure; and
- **trust** identifies components and assumptions whose failure can invalidate the claim.

The distinction matters. A file can have a correct mode and ACL while a privileged parser is exploitable. Two containers can see different process trees while sharing one vulnerable kernel. A seccomp filter can block `mount()` while the process still has a writable descriptor for a sensitive file. No single control turns an arbitrary program into a secure system.

![Protection, isolation, kernel enforcement, and audit form an end-to-end security contract.](assets/security-protection-trust-boundaries.svg){fig-alt="Layered security contract from policy and threat model through reference mediation, isolation, kernel and hardware enforcement, to audit and recovery feedback." width="96%"}

*Figure: original explanatory diagram based on the [NIST reference-monitor definition](https://csrc.nist.gov/glossary/term/reference_monitor), [Saltzer and Schroeder's protection principles](https://www.cs.virginia.edu/~evans/cs551/saltzer/), and the [Linux kernel security documentation](https://docs.kernel.org/security/index.html).*

The chapter continues the running pipeline:

```bash
cat input.txt | grep kernel > result.txt
```

The shell resolves the two pathnames, opens the files, creates a pipe, forks, rearranges descriptors, and executes `cat` and `grep`. Protection determines whether the shell may traverse the directories and open the inodes. Descriptor inheritance determines which object authority reaches each child. Memory and syscall controls limit what a compromised utility can do. Namespaces change its view; cgroups bound its resource use; audit records preserve evidence of important decisions.

A useful mental model is:

> A security-sensitive operation is safe only when the kernel mediates a stable subject, a stable object, an explicit operation, and the correct policy under a stated threat model, while every component trusted by that decision behaves as assumed.

This chapter focuses on local operating-system controls. Cryptographic protocol design, remote identity federation, network firewalls, malware analysis and distributed authorization belong to other paths, although their OS-facing credentials, sockets, keys and processes still use the mechanisms developed here. Virtual machines and container packaging are deferred to the next chapter; here, namespaces and cgroups appear as protection mechanisms rather than deployment products.

### **Protection, Security, and Trust**

A **protection policy** describes allowed relations such as “processes in the report role may read input objects and write only into the report output directory.” A **protection mechanism** supplies the concrete check: UID comparison, permission bits, an ACL entry, an LSM label rule, a kernel-held capability, or a validated descriptor.

Security is broader because an attacker chooses inconvenient paths through the mechanism. They may exploit a parser before the permission check, replace a pathname between check and use, inherit a descriptor that bypasses later lookup, force a resource limit to fail at a dangerous point, or manipulate logs after compromise. Security arguments therefore start with an explicit model rather than a list of enabled features.

Several properties are related but independent:

| Property | Security question | Representative OS failure |
|---|---|---|
| Confidentiality | Can an unauthorized subject learn the data? | Secret descriptor inherited across `execve()` |
| Integrity | Can an unauthorized subject alter data or control state? | Symlink race redirects privileged output |
| Availability | Can legitimate work continue within its service objective? | Unbounded process creation exhausts PIDs or memory |
| Authenticity | Is the principal or object really the one claimed? | Authorization uses a recycled name instead of a stable handle |
| Accountability | Can an action be attributed with trustworthy evidence? | Audit backlog overflows and the loss is ignored |
| Least authority | Does each component hold only necessary power for only as long as needed? | Parser runs permanently as root with every descriptor open |

Protection mechanisms are usually **fail-safe** when absence of an explicit grant causes denial. This does not mean every error should terminate the whole machine. It means that malformed policy, unsupported kernel features, ambiguous identity and failed sandbox installation must not silently produce a more privileged execution than intended.

#### **Threat Models and the Trusted Computing Base**

A **threat model** states the security problem before choosing controls. At minimum, it identifies:

1. **assets**: data, control over a process or device, keys, service capacity, and evidence;
2. **subjects and adversaries**: which processes, users, inputs or administrators may behave maliciously;
3. **entry points**: syscalls, IPC, executable loading, files, devices, sockets, configuration and recovery paths;
4. **required properties**: what must remain confidential, integral, available and attributable;
5. **adversary capabilities**: code execution, race scheduling, credential possession, local physical access, kernel access, or only crafted input; and
6. **assumptions and exclusions**: trusted hardware, boot chain, kernel, policy loader, clock, storage, administrator, or external identity provider.

![A threat model defines assets, adversary power, entry points, properties, assumptions, and the trusted computing base.](assets/threat-model-tcb.svg){fig-alt="Threat model boxes for assets, adversary capabilities, entry points and required properties connected to a trusted computing base of hardware, kernel, boot policy, privileged helpers and security state." width="97%"}

*Figure: original explanatory diagram based on the [NIST reference-monitor model](https://csrc.nist.gov/glossary/term/reference_monitor), [NIST SP 800-53 terminology](https://csrc.nist.gov/pubs/sp/800/53/r5/upd1/final), and [seL4's discussion of verified kernel assumptions](https://sel4.systems/About/FAQ.html).*

Threat models should distinguish attacker levels because the same control has different meaning under each one:

| Adversary model | Example capability | Useful controls | Controls no longer sufficient |
|---|---|---|---|
| Malformed input | Supplies bytes to a normal utility | Validation, memory safety, least privilege | Input validation alone if parser has memory corruption |
| Unprivileged code execution | Runs arbitrary code under one UID | DAC/ACL, namespaces, seccomp, Landlock, cgroups | Application checks inside the compromised process |
| Compromised service credential | Uses a service's UID, groups and tokens | Per-object policy, privilege separation, short-lived handles | Controls that equate UID with one trustworthy program |
| Privileged user-space compromise | Controls a root daemon or broker | Small privilege domains, MAC, verified protocols, protected audit | Ordinary root-bypassable DAC |
| Kernel compromise | Executes in the host kernel | Hypervisor/secure monitor boundaries, measured boot, hardware isolation | Same-kernel process/container isolation |
| Physical/firmware compromise | Alters memory, firmware or storage directly | Secure boot, device identity, encryption, tamper controls | Pure software permission checks on that machine |

The **Trusted Computing Base (TCB)** is the set of hardware, software and configuration whose correct behavior is necessary for the security claim. It is not a list of components that seem reputable; it is the dependency closure of the claim. If a root helper can rewrite an ACL, it belongs to the TCB for that ACL's enforcement. If policy arrives from a configuration service, that service and its authentication path may also be trusted.

Trust and trustworthiness are different. A large privileged parser may be trusted because the design depends on it, while remaining difficult to test and therefore not very trustworthy. Good design reduces both **TCB size** and **TCB interface area**:

- move complex, attacker-facing parsing into unprivileged processes;
- keep privileged protocols narrow, typed and bounded;
- remove unnecessary kernel modules, syscalls, devices and inherited descriptors;
- make security state immutable or monotonically more restrictive after initialization;
- centralize decision paths so bypasses are difficult to create; and
- make assumptions observable through boot measurements, policy hashes, configuration inventory and audit.

Formal verification can raise assurance about a precisely modeled component, but it does not remove assumptions about hardware, compiler, specification, deployment policy or components outside the proof. Likewise, “the kernel enforces it” is only meaningful if every path reaches the check and the kernel is inside the accepted TCB.

### **Protection Domains and the Reference Monitor**

A **protection domain** is the authority available to an executing subject at a particular time. It can be represented conceptually as a set of object-operation pairs:

$$
D(p,t) = \{(o, r) \mid \text{process } p \text{ may perform right } r
\text{ on object } o \text{ at time } t\}.
$$

The domain is not identical to a username. A process's domain also depends on supplementary groups, Linux capability sets, user and mount namespaces, LSM labels, open descriptors, mapped memory, IPC endpoints, tokens, current sandbox rules and cgroup placement. Two processes with the same effective UID may hold different descriptors and therefore different practical authority.

A domain may change through carefully mediated transitions:

- `execve()` can apply set-user-ID metadata, file capabilities and LSM transitions;
- a process can drop IDs, groups and capabilities;
- a broker can pass a descriptor through a Unix-domain socket;
- entering a namespace changes the view and scope in which credentials operate;
- installing seccomp or Landlock policy monotonically restricts later behavior; and
- revoking an ACL may affect future opens while existing descriptors remain usable according to object and filesystem semantics.

The **reference monitor** is the design model for enforcing these relations. NIST summarizes three requirements: the mechanism is always invoked, resistant to tampering, and small enough to analyze and test. The concrete implementation is often distributed across syscall entry, VFS permission checks, object-specific kernel code, LSM hooks, namespace ownership rules and hardware page protection, but it must behave like a non-bypassable decision path.

![A reference monitor resolves subject and object contexts, evaluates policy, and mediates every protected operation.](assets/reference-monitor-mediation.svg){fig-alt="Subject request crosses a reference validation mechanism that resolves task credentials and a stable object, evaluates policy, and reaches the object only on allow; properties are always invoked, tamper resistant and verifiable." width="97%"}

*Figure: original explanatory diagram based on the [NIST reference monitor definition](https://csrc.nist.gov/glossary/term/reference_monitor) and Saltzer and Schroeder's principles of [complete mediation and economy of mechanism](https://www.cs.virginia.edu/~evans/cs551/saltzer/).*

An authorization decision can be modeled as:

$$
\operatorname{decision} = P(S, O, R, E),
$$

where $S$ is the stable subject context, $O$ the stable object context, $R$ the requested operation, and $E$ relevant environment such as namespace ownership, mount flags, time or policy generation. An `ALLOW` decision should authorize **that operation on that resolved object**, not an object that happens to have the same mutable pathname later.

Complete mediation does not forbid caching. A dentry, credential pointer or access decision can be cached if the cache key captures all security-relevant state and policy changes invalidate or version the result. A cache keyed only by pathname, for example, is unsafe if mount namespaces, symlinks, credentials or labels can differ.

Three questions belong at different layers:

| Question | Mechanism | Failure example |
|---|---|---|
| Authentication: who or what is the principal? | Login/session setup, process credentials, peer credentials, keys | Reused numeric UID interpreted in the wrong user namespace |
| Authorization: may this operation occur? | DAC, ACL, capabilities, LSM policy, handle rights | Check applies to pathname A but operation reaches inode B |
| Accountability: what decision and effect occurred? | Kernel audit, application event IDs, protected log collection | Log says “open attempted” but omits result or object identity |

A **confused deputy** appears when a more privileged component uses its ambient authority on behalf of an untrusted requester without binding the requester's authority or intent to the operation. Passing an already-open, rights-reduced descriptor is often safer than asking a root process to reopen an attacker-provided path. The descriptor makes both the object and delegated operation more explicit.

### **Identity, Users, Groups, and Credentials**

An operating system authorizes **running subjects**, not human names written in a configuration file. Login infrastructure maps a human or service identity into numeric IDs, groups, tokens and session state; the kernel then carries a credential object on each task and compares that subjective context with protected objects.

Linux separates several UID/GID roles:

| Credential | Main purpose |
|---|---|
| Real UID/GID | Origin/accounting identity inherited from the launching context |
| Effective UID/GID | Traditional identity used for many permission and privilege decisions |
| Saved set-user-ID/GID | Supports controlled dropping and possible restoration after `execve()` transitions |
| FSUID/FSGID | Filesystem-check identity, normally equal to effective IDs |
| Supplementary groups | Additional group memberships considered by file and other checks |
| Audit login UID (AUID) | Tracks the original login identity across later effective-ID changes |

Modern Linux stores task credentials in a reference-counted `struct cred`. Published credential structures are treated as immutable: a task prepares a modified copy and commits a pointer replacement. This avoids readers observing a half-updated combination of UID, groups, keys and security state.

![Linux compares a task's credential context with the security context of a filesystem, process, IPC, socket, or kernel object.](assets/linux-credential-context.svg){fig-alt="Linux task_struct points to immutable struct cred containing IDs, supplementary groups, capability sets, keyrings and LSM context, compared with the target object's owner, ACL, label, namespace and type." width="98%"}

*Figure: original explanatory diagram based on the official [Linux credentials documentation](https://docs.kernel.org/security/credentials.html), [`credentials(7)`](https://man7.org/linux/man-pages/man7/credentials.7.html), and [`capabilities(7)`](https://man7.org/linux/man-pages/man7/capabilities.7.html).*

The comparison is operation-specific. VFS pathname access normally uses filesystem IDs, group membership, inode mode/ACL, relevant capabilities, mount state and LSM hooks. Signaling another process compares task credentials and namespace relationships. Opening a socket, loading a BPF program, tracing another task or configuring a device follows different rules.

Credentials also have lifetime semantics:

- `fork()` creates a child initially using the same credential values;
- credentials are per thread at the kernel level, although POSIX thread libraries coordinate changes so a process appears to change them consistently;
- `execve()` preserves many identity fields but may derive a new effective identity and capability sets from executable metadata and policy;
- a file's kernel open-file object records opening credentials for operations that must retain the opener's context; and
- Unix-domain sockets can carry kernel-supplied peer credentials, avoiding trust in self-reported PID/UID strings.

Numeric IDs have meaning only in context. User namespaces map an inside ID to a different outside ID. Network filesystems may translate identities through server policy. Deleting and reusing a local UID can make old files appear owned by a new account. Security state should therefore use stable lifecycle management and namespace-aware interpretation, not only display names.

<details>
<summary><strong>Inspect task, file, namespace, and peer credential context on Linux</strong></summary>

```bash
# Human-readable real/effective IDs and supplementary groups.
id

# Kernel task status: real/effective/saved/fs IDs, groups, capabilities,
# no_new_privs, seccomp mode, and namespace-relevant state.
grep -E '^(Uid|Gid|Groups|Cap(Inh|Prm|Eff|Bnd|Amb)|NoNewPrivs|Seccomp):' \
  /proc/self/status

# The numeric owner, group, mode, inode and filesystem identity of an object.
stat -c 'inode=%i uid=%u gid=%g mode=%A (%a) dev=%D' ./input.txt

# Show each path component and the permissions that govern traversal.
namei -l ./input.txt

# Namespace handles identify the views in which IDs and objects are interpreted.
readlink /proc/self/ns/{user,mnt,pid,net,ipc,uts,cgroup}

# On systems with libcap tools, decode the current capability sets.
getpcaps $$
capsh --print
```

`/proc/self/status` exposes hexadecimal capability masks; `getpcaps`/`capsh` decode them when installed. These commands describe current state, not why a particular LSM or object-specific check allowed an operation. Audit records, policy tools and the failing syscall are still needed for that diagnosis.

</details>

### **Access-Control Models**

An **access-control model** supplies the vocabulary for a policy; an enforcement mechanism realizes that model for concrete kernel objects. Common models include:

- **discretionary access control (DAC)**: an owner or delegated administrator can grant access, as with Unix mode bits and POSIX ACLs;
- **mandatory access control (MAC)**: centrally managed labels and rules constrain even object owners, as in SELinux-style type enforcement;
- **role-based access control (RBAC)**: permissions attach to organizational roles and users activate authorized roles;
- **attribute-based access control (ABAC)**: a rule considers attributes of subject, object, operation and environment; and
- **capability-based access control**: possession of an unforgeable object reference carrying rights is evidence of authority.

Real systems compose these models. A VFS operation may pass DAC and capability checks but be denied by an LSM. A container's UID 0 may have capabilities only in a child user namespace. A broker may use identity policy once, then delegate a narrow descriptor so the worker no longer needs ambient pathname authority.

The conceptual foundation is an **access matrix**. Rows are subjects or protection domains, columns are objects, and each cell is a set of rights:

$$
A[s,o] \subseteq \{\text{read},\text{write},\text{execute},\text{signal},
\text{map},\text{delegate},\ldots\}.
$$

The matrix is normally huge and sparse, so implementations store projections or derive cells from policy.

![The access matrix can be stored by object as an ACL or by subject as a capability list.](assets/access-matrix-projections.svg){fig-alt="Sparse shell pipeline access matrix with rows for shell, cat and grep and columns for input, output and pipe; object column projects to an ACL and grep row projects to capabilities." width="98%"}

*Figure: original explanatory diagram based on the access-matrix treatment in [Cambridge Operating Systems notes](https://www.cl.cam.ac.uk/teaching/1819/OpSystems/pdf/ia-os.pdf), [seL4 capability documentation](https://docs.sel4.systems/Tutorials/capabilities.html), and [`acl(5)`](https://man7.org/linux/man-pages/man5/acl.5.html).*

Policy design also needs administrative operations: who may grant, delegate, copy, attenuate and revoke a right? A read permission and permission to grant that read permission are different rights. Revocation may be immediate for future name-based checks but delayed for existing handles, memory mappings, cached tokens or replicas.

#### **Access Matrices and Permission Bits**

Unix mode bits are a compact DAC approximation. An inode stores an owner UID, group GID and three `rwx` triplets. During an ordinary check, the kernel chooses one class rather than combining all three:

1. if the filesystem UID matches the inode owner, use owner bits;
2. otherwise, if filesystem GID or any supplementary group matches the inode group, use group bits;
3. otherwise use other bits; and
4. then apply relevant capability overrides, mount constraints and LSM policy.

For a regular file, `r`, `w` and `x` mean reading bytes, modifying bytes and executing the object. For a directory they mean listing names, modifying directory entries, and **searching/traversing names**. This distinction explains several apparently surprising outcomes.

![Pathname resolution checks search permission on each directory before applying the final file operation.](assets/unix-permission-path-walk.svg){fig-alt="Process opens /srv/project/result.txt by checking execute-search on root, srv and project directories, then write on the final file; separate boxes compare file and directory rwx meanings." width="98%"}

*Figure: original explanatory diagram based on [`path_resolution(7)`](https://man7.org/linux/man-pages/man7/path_resolution.7.html), [`inode_permission`](https://docs.kernel.org/filesystems/vfs.html), and [`capabilities(7)`](https://man7.org/linux/man-pages/man7/capabilities.7.html).*

Important consequences include:

- read permission on a directory can reveal names but, without search permission, does not permit normal resolution through those names;
- search without read can permit access to a known name while preventing a directory listing;
- creating or removing a name generally requires write **and** search permission on the parent directory, not write permission on the target file;
- the sticky bit on a shared directory such as `/tmp` further restricts who can remove or rename entries;
- the set-group-ID bit on a directory can make new children inherit the directory's group, supporting collaborative trees; and
- execute permission on a script does not replace read/interpretation rules, shebang handling, mount `noexec`, interpreter access or LSM policy.

When creating an object without a default ACL, requested mode is filtered by the process umask:

$$
\text{created permission bits} = \text{requested bits} \;\&\; \sim\text{umask}.
$$

`umask` removes permissions; it does not add them. The result can be further affected by default ACL inheritance, set-group-ID directories and filesystem policy. Security-sensitive code should pass an intentionally narrow creation mode and not rely on a globally convenient umask.

<details>
<summary><strong>Trace Unix permission and directory-entry behavior</strong></summary>

```bash
# Read all path components, then inspect the final inode and any ACL.
namei -l /srv/project/result.txt
stat -c '%A %a owner=%U(%u) group=%G(%g) inode=%i' /srv/project/result.txt
getfacl -p /srv/project/result.txt 2>/dev/null

# Display the current creation mask without changing it.
umask -S

# Symbolic examples; run only on a disposable test tree you own.
# chmod 2770 project       # group-collaborative directory; inherit group
# chmod 1777 shared        # world-writable plus sticky deletion rule
# install -m 0640 /dev/null project/result.txt
```

Use `namei -l` when a final inode appears permissive but `open()` returns `EACCES`: a parent directory may deny search. `ls -l` shows only the compact mode representation; `getfacl` is needed when extended ACL entries and masks change effective rights.

</details>

#### **Access-Control Lists**

An **access-control list (ACL)** stores an object-centric list of principals and rights. POSIX ACLs extend the three Unix classes with named-user and named-group entries while preserving compatibility with mode bits. A directory can also hold a **default ACL** that initializes children created beneath it.

The non-obvious element is the **ACL mask**. It limits the effective permissions of named users, the owning group entry and named groups. It does not limit the owner entry or `other`. Therefore an entry that displays `user:bob:rw-` may be effectively read-only when the mask is `r--`.

![The POSIX ACL algorithm checks owner, named user, matching groups and other, with a mask limiting named-user and group classes.](assets/posix-acl-decision.svg){fig-alt="POSIX ACL decision tree from owner to named user, matching groups and other; example ACL shows named user and editors group rw entries reduced to read by mask r--." width="97%"}

*Figure: original explanatory diagram following the normative access-check and inheritance algorithm in [`acl(5)`](https://man7.org/linux/man-pages/man5/acl.5.html).*

The simplified decision order is:

```text
POSIX-ACL-CHECK(task, inode, requested)
    if task.fsuid == inode.owner:
        effective <- ACL_USER_OBJ
    else if a named ACL_USER entry matches task.fsuid:
        effective <- matching_user AND ACL_MASK
    else if inode.group or any named ACL_GROUP matches task groups:
        effective <- UNION(all matching group entries) AND ACL_MASK
    else:
        effective <- ACL_OTHER

    allow only if requested is a subset of effective
```

A matching named-user or group class that lacks a requested right does not simply fall through to a more generous `other` entry. The matching class determines the result. This prevents policy from becoming more permissive merely because a principal was identified more specifically.

Default ACLs and umask interact during creation. If the parent has a default ACL, that ACL is inherited and then restricted by the mode requested by `open()`, `mkdir()` or related calls. Without a default ACL, mode and umask construct the minimal owner/group/other ACL. Changing mode bits with `chmod` also changes corresponding ACL entries and can tighten the mask.

<details>
<summary><strong>Create and inspect a POSIX ACL on a disposable tree</strong></summary>

```bash
mkdir -p ./acl-demo/project
printf 'draft\n' > ./acl-demo/project/report.txt

# Grant named user bob rw in the entry, then cap the group class at read.
setfacl -m u:bob:rw-,m::r-- ./acl-demo/project/report.txt
getfacl ./acl-demo/project/report.txt

# Seed future children with a group-collaboration default ACL.
setfacl -m d:u::rwx,d:g::rwx,d:o::---,d:m::rwx ./acl-demo/project
touch ./acl-demo/project/new.txt
getfacl ./acl-demo/project/new.txt

# Remove extended entries when finished.
setfacl -b ./acl-demo/project/report.txt
setfacl -k ./acl-demo/project
```

The first `getfacl` output should annotate Bob's effective right as `r--` because the mask is a ceiling. Tool availability and ACL support depend on the filesystem and mount; failure to set the required policy should be treated explicitly rather than ignored.

</details>

ACLs make “who may access this object?” easy to inspect and make object-specific revocation straightforward. They can become difficult to administer across many objects, identities and inherited trees. Groups, roles, labels and policy generation tools reduce duplication, but their expansion must remain reviewable.

#### **Capabilities**

The word **capability** has two related but distinct operating-system meanings:

1. An **object capability** is an unforgeable token that names a particular object and carries rights over it. seL4 capabilities are a direct example; Unix descriptors are capability-like kernel handles when the surrounding process cannot reacquire broader authority through ambient names.
2. A **Linux capability** is one unit split from traditional root privilege, such as `CAP_NET_BIND_SERVICE`, `CAP_CHOWN` or `CAP_DAC_READ_SEARCH`. It authorizes classes of operations rather than naming one specific file or socket.

Confusing the two leads to bad designs. Giving a worker Linux `CAP_DAC_OVERRIDE` is broad ambient power over many filesystem objects. Giving it one write-only descriptor for `result.txt` is object-specific authority.

![A privileged broker can delegate a rights-reduced object handle without giving a worker authority to resolve unrelated paths.](assets/capability-delegation.svg){fig-alt="Broker opens the intended result inode and passes a kernel-managed write-only file descriptor to an unprivileged worker; the worker cannot forge authority for an unrelated secret." width="97%"}

*Figure: original explanatory diagram based on the [seL4 capability tutorial](https://docs.sel4.systems/Tutorials/capabilities.html), the [seL4 capability definition](https://sel4.systems/About/FAQ.html), and Unix descriptor passing in [`unix(7)`](https://man7.org/linux/man-pages/man7/unix.7.html).*

An object capability normally provides:

- **designation**: it identifies the object without a second ambient name lookup;
- **authority**: it states permitted operations;
- **unforgeability**: untrusted code cannot manufacture a valid token for another object;
- **delegation**: a holder can pass or derive a reduced capability if policy permits; and
- **attenuation**: rights can be narrowed, for example from read-write to read-only.

Revocation is the difficult direction. Deleting one ACL entry can affect future checks on an object, while recalling every copied capability may require indirection, versioned handles, a revocation tree, lease expiry or destroying the referenced object. Capability systems therefore design derivation and revocation together.

Unix descriptors illustrate both the strength and caveat. After `open()`, descriptor operations target the resolved open file, so pathname replacement cannot retarget the handle. `SCM_RIGHTS` can pass that handle to another process. But a descriptor is only part of a capability-safe design if the receiver lacks broader ambient authority through unrestricted directories, `/proc` interfaces, inherited descriptors, devices or privileged syscalls.

Linux capability sets refine superuser privilege:

| Set | Meaning |
|---|---|
| Permitted | Upper set the thread may make effective under capability rules |
| Effective | Capabilities currently consulted by kernel permission checks |
| Inheritable | Capabilities eligible to participate in selected `execve()` transitions |
| Bounding | Ceiling on capabilities that can be gained through file capability transitions |
| Ambient | Capabilities preserved across ordinary non-privileged `execve()` under strict invariants |

File `security.capability` metadata can grant selected capability sets when an executable is loaded. User namespaces scope capability meaning: a process can be UID 0 with many effective capabilities in a child user namespace while lacking equivalent power over resources governed by the initial user namespace.

Capabilities reduce privilege only when the selected units are genuinely narrow. `CAP_SYS_ADMIN` authorizes a very large and evolving collection of operations and often acts as “almost root.” Prefer object handles, dedicated brokers and narrower capabilities where possible; drop the bounding and ambient sets when they are not needed.

<details>
<summary><strong>Inspect Linux capabilities without granting new privilege</strong></summary>

```bash
# Process capability masks and irreversible no-new-privileges state.
grep -E '^(Cap(Inh|Prm|Eff|Bnd|Amb)|NoNewPrivs):' /proc/self/status
getpcaps $$ 2>/dev/null || true

# File capabilities already present on selected executables.
getcap /usr/bin/ping /usr/bin/passwd 2>/dev/null

# Decode a hexadecimal mask with capsh when libcap tools are installed.
capsh --decode="$(awk '/^CapEff:/ {print "0x" $2}' /proc/self/status)"

# Administrative example only; do not run on production binaries casually:
# sudo setcap cap_net_bind_service=ep ./small-server
# getcap ./small-server
# sudo setcap -r ./small-server
```

File capabilities alter the `execve()` security transition and must be managed like set-user-ID metadata. Copying, archiving, mounting with `nosuid`, user namespaces and filesystem extended-attribute support can change whether they are retained or honored.

</details>

| Design question | Object ACL | Object capability | Linux capability |
|---|---|---|---|
| Primary key | Object then identity/group | Unforgeable object reference | Privileged operation class |
| Delegation | Edit ACL/group or use broker | Pass/derive a token or handle | Inherit/transfer under capability rules |
| Ambient authority | Identity can reach many named objects | Can be minimized to held references | Usually applies to many kernel objects |
| Revocation | Remove entry for future checks | Requires revocation design/indirection | Drop set, bound inheritance, stop process |
| Best use | Stable identity-centric sharing | Fine-grained least-authority collaboration | Replace selected traditional root powers |

### **Privilege and Least Authority**

**Privilege** is authority that ordinary execution does not possess: bypassing DAC, changing identities, administering mounts or networks, tracing another task, loading kernel code, or controlling protected devices. Least privilege is not merely “avoid root.” It asks that each component hold the smallest object set, operation set, time interval and delegation power needed for one responsibility.

Saltzer and Schroeder's classic principles remain practical design tests:

| Principle | Operating-system interpretation |
|---|---|
| Economy of mechanism | Keep the TCB and privileged protocol small |
| Fail-safe defaults | Require an explicit grant; sandbox setup failure denies launch |
| Complete mediation | Check every effective access path and invalidate stale decisions |
| Open design | Depend on protected keys/state, not secrecy of the mechanism |
| Separation of privilege | Require independent conditions or split authority across components |
| Least privilege | Minimize rights and drop them as soon as initialization ends |
| Least common mechanism | Reduce shared writable state and shared privileged services |
| Psychological acceptability | Make the safe workflow the ordinary, understandable workflow |

Least authority can be considered along four dimensions:

$$
\text{exposure} \propto
|\text{objects}| \times |\text{operations}| \times
\text{privileged time} \times \text{reachable input surface}.
$$

This is not a risk formula with calibrated units; it is a review heuristic. Reducing any factor can reduce the blast radius of compromise. A worker that holds one output descriptor for ten milliseconds is easier to reason about than a daemon that can reopen every path indefinitely.

**Privilege separation** divides a service into protection domains. A small broker retains only the rights needed to perform sensitive operations; complex parsers and workers run without those rights. The broker authenticates the caller, validates a narrow request, resolves the object safely, and returns a rights-reduced handle or bounded result.

![Privilege separation confines ambient authority to a small broker behind a narrow IPC protocol.](assets/privilege-separation.svg){fig-alt="Unprivileged parser and worker communicate using a bounded protocol with a small privileged broker that alone can open protected resources and pass minimal handles." width="97%"}

*Figure: original explanatory diagram based on Saltzer and Schroeder's [least privilege and separation of privilege](https://www.cs.virginia.edu/~evans/cs551/saltzer/), the Linux [credentials model](https://docs.kernel.org/security/credentials.html), and descriptor delegation through [`unix(7)`](https://man7.org/linux/man-pages/man7/unix.7.html).*

The broker must not become a confused deputy. A request such as `open(path_from_client)` implicitly combines the broker's ambient privilege with the client's chosen object. A safer request might select a documented operation and a relative name under a pre-opened directory, with the broker applying `openat2()` constraints and returning a descriptor of fixed access mode.

Other practical rules include:

- initialize privileged resources before accepting untrusted input;
- close every descriptor not included in the domain's contract and use `O_CLOEXEC` by default;
- avoid privileged use of attacker-controlled environment variables, search paths, locale data and configuration;
- authenticate IPC peers with kernel-supplied credentials or cryptographic channels rather than self-reported IDs;
- make protocol lengths, counts and operation enums bounded;
- treat broker timeout, crash and restart semantics as part of the security policy; and
- audit both accepted and rejected high-impact requests without placing secrets in logs.

#### **Privilege Separation and Controlled Elevation**

Some tasks genuinely require authority unavailable to the caller. **Controlled elevation** creates a narrow, reviewable transition rather than running an entire application with permanent privilege.

Traditional Unix set-user-ID execution sets the effective UID to the executable owner's UID. Linux file capabilities can grant selected operation classes instead. Tools such as `sudo`, service managers and desktop authorization brokers add policy, authentication, environment construction, logging and lifecycle management around privileged execution.

`execve()` is the key transition point. The kernel considers executable owner/mode, file capabilities, mount `nosuid`, ptrace state, user namespace, capability bounding set, securebits and LSM policy before constructing the new credential set.

![execve can apply set-user-ID or file-capability metadata, while no_new_privs prevents the execution from adding privilege.](assets/exec-privilege-transition-animated.svg){fig-alt="Animated execution path from an ordinary UID 1000 process through execve loader checks to ordinary, setuid-root or file-capability results; no_new_privs blocks the privilege-gaining branches." width="98%"}

*Figure: original animated diagram based on the Linux [`no_new_privs` documentation](https://docs.kernel.org/userspace-api/no_new_privs.html), [`execve(2)`](https://man7.org/linux/man-pages/man2/execve.2.html), and [`capabilities(7)`](https://man7.org/linux/man-pages/man7/capabilities.7.html).*

The `no_new_privs` task flag is especially useful for sandbox launchers. Once set, it is inherited across `fork()`, `clone()` and `execve()` and cannot be unset. An `execve()` then promises not to grant authority unavailable without the execution: setuid/setgid bits do not raise IDs and file capabilities do not add to the permitted set. It does **not** prevent an already privileged process from changing IDs through other authorized calls or using descriptors it already holds.

A typical one-way privilege-drop sequence is:

1. create only the required listeners, files, namespaces and IPC endpoints;
2. apply ownership, mode and object policy while authorized;
3. remove supplementary groups that should not survive;
4. set the final GID and UID, including saved IDs, so privilege cannot be regained;
5. clear effective/permitted/ambient capabilities and reduce the bounding set;
6. set `no_new_privs` and install sandbox policy;
7. close unintended descriptors and sanitize `argv`, environment and working directory; and
8. verify the resulting IDs and capability state before processing attacker-controlled data.

Order matters. Dropping UID before clearing groups can make `setgroups()` fail, accidentally retaining group authority. Setting `no_new_privs` too late can allow an intermediate `execve()` transition. Keeping a privileged descriptor open can defeat a correct UID drop.

<details>
<summary><strong>Linux C example: open one output, then irreversibly drop IDs before exec</strong></summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <fcntl.h>
#include <grp.h>
#include <pwd.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/prctl.h>
#include <sys/types.h>
#include <unistd.h>

static void die(const char *operation) {
    perror(operation);
    exit(EXIT_FAILURE);
}

static void drop_ids(uid_t uid, gid_t gid) {
    // Supplementary groups carry authority independently of the primary GID.
    if (setgroups(0, NULL) == -1) die("setgroups");

    // Set real, effective, and saved IDs together so this process cannot
    // restore the old privileged effective IDs later.
    if (setresgid(gid, gid, gid) == -1) die("setresgid");
    if (setresuid(uid, uid, uid) == -1) die("setresuid");

    // Prevent the following exec from gaining privilege through setuid bits,
    // setgid bits, or file capabilities. The flag is irreversible.
    if (prctl(PR_SET_NO_NEW_PRIVS, 1, 0, 0, 0) == -1) {
        die("PR_SET_NO_NEW_PRIVS");
    }

    if (getuid() != uid || geteuid() != uid ||
        getgid() != gid || getegid() != gid) {
        fputs("credential verification failed\n", stderr);
        exit(EXIT_FAILURE);
    }
}

int main(int argc, char **argv) {
    if (argc != 4) {
        fprintf(stderr, "usage: %s USER OUTPUT PATTERN\n", argv[0]);
        return EXIT_FAILURE;
    }

    // Resolve account information before dropping access to identity services.
    errno = 0;
    struct passwd *account = getpwnam(argv[1]);
    if (account == NULL) {
        if (errno != 0) perror("getpwnam");
        else fprintf(stderr, "unknown user: %s\n", argv[1]);
        return EXIT_FAILURE;
    }
    uid_t target_uid = account->pw_uid;
    gid_t target_gid = account->pw_gid;

    // The privileged phase acquires one intentionally narrow object handle.
    // A production broker should resolve an administrator-controlled path
    // relative to a trusted directory descriptor, not accept an arbitrary path.
    int output = open(argv[2], O_WRONLY | O_APPEND | O_CLOEXEC);
    if (output == -1) die("open output");

    drop_ids(target_uid, target_gid);

    // dup2 creates stdout without close-on-exec. Handle the unusual case in
    // which open() reused descriptor 1 because the caller had closed stdout.
    if (output == STDOUT_FILENO) {
        int descriptor_flags = fcntl(output, F_GETFD);
        if (descriptor_flags == -1) die("F_GETFD stdout");
        if (fcntl(output, F_SETFD, descriptor_flags & ~FD_CLOEXEC) == -1) {
            die("clear FD_CLOEXEC on stdout");
        }
    } else {
        if (dup2(output, STDOUT_FILENO) == -1) die("dup2");
        if (close(output) == -1) die("close output");
    }

    // Use a fixed executable and a minimal environment instead of PATH lookup
    // or inherited loader variables from a privileged caller.
    char *const child_argv[] = {"/usr/bin/grep", "--", argv[3], NULL};
    char *const child_env[] = {"PATH=/usr/bin:/bin", "LANG=C", NULL};
    execve(child_argv[0], child_argv, child_env);
    die("execve grep");
}
```

Compile on Linux:

```bash
cc -std=c11 -Wall -Wextra -O2 drop_exec.c -o drop_exec
# Example launch requires an already-created output and appropriate privilege:
# cat input.txt | sudo ./drop_exec nobody /safe/output/result.txt kernel
```

This example focuses on ID ordering, environment control and descriptor delegation. A production launcher should also clear and verify all Linux capability sets and the bounding/ambient sets, close every unrelated descriptor (for example with `close_range()`), apply object and syscall policy, avoid an attacker-controlled output pathname, handle namespaces explicitly, and define audit/error behavior. “Changed EUID” alone is not a complete privilege drop.

</details>

Privilege restoration is sometimes deliberately required by legacy designs using saved IDs. That makes the process permanently more dangerous: any memory-corruption bug after the apparent drop may regain authority. A broker process with a narrow IPC protocol is generally easier to reason about than repeatedly toggling one large process between privileged and unprivileged modes.

### **Defending the System Call Boundary**

The system call boundary is where untrusted process state becomes a kernel request. Ring transition hardware protects kernel memory, but it does not make arguments valid. Kernel code must treat syscall numbers, pointers, lengths, flags, handles and process state as adversarial.

A robust syscall path normally performs these steps:

1. decode the correct architecture and ABI;
2. reject unknown flags, invalid combinations and non-canonical values;
3. copy required user data into kernel-owned memory exactly when needed;
4. validate lengths before arithmetic and allocation;
5. resolve handles under appropriate lifetime/refcount protection;
6. authorize the specific operation on the resolved object;
7. reserve resources or establish rollback before visible mutation;
8. perform the operation while respecting concurrency and cancellation; and
9. copy bounded results back, report partial progress and preserve a meaningful error.

Common failures map to concrete defenses:

| Boundary error | Example consequence | Defensive pattern |
|---|---|---|
| Trust user pointer after validation | Another thread changes pointed-to data | Copy once or pin under documented semantics; validate the copy |
| Integer overflow in `count * size` | Undersized allocation and overwrite | Checked arithmetic and upper bounds before allocation |
| Accept unknown flag bits | Future/accidental behavior bypasses policy | Reject `flags & ~KNOWN_FLAGS` |
| Use numeric ID/descriptor without lifetime check | Reuse targets a different object | Hold a reference or generation-protected handle |
| Authorize pathname, then resolve again | TOCTOU redirects operation | Resolve once atomically; continue through descriptor |
| Mutate before all failure checks | Partial privileged side effect | Stage, reserve, commit, or define rollback/idempotence |
| Omit resource accounting | Attacker exhausts kernel memory or table | Per-user/cgroup/global limits and bounded queues |
| Return inconsistent errors | Caller retries unsafe or leaks state | Stable error contract; avoid sensitive distinction when necessary |

Kernel APIs often expose **handles** because they bind identity and lifetime. A file descriptor, PID descriptor, namespace descriptor or device queue handle is resolved by the kernel into a reference-counted object. The handle can still be misused, but it prevents later name reuse from silently selecting another object.

#### **Input Validation, Handles, and Time-of-Check-Time-of-Use**

A **time-of-check to time-of-use (TOCTOU)** bug occurs when security-relevant state can change between validation and effect. Filesystem pathnames are a classic source because each component can be renamed, replaced by a symlink, hidden by a mount or interpreted in another namespace.

Vulnerable privileged code often resembles:

```c
if (access(path, W_OK) == 0) {       // checks one resolution, often real IDs
    int fd = open(path, O_WRONLY);   // resolves the pathname again
    /* privileged write through fd */
}
```

Even if both calls are individually correct, no invariant says they target the same inode. Another thread or process can replace a component between calls. Locks inside this program do not stop external namespace mutation unless the lock also controls every party capable of changing the path.

![A separate pathname check can race with attacker replacement, while constrained openat2 resolution returns one stable descriptor.](assets/toctou-handle-resolution-animated.svg){fig-alt="Animated top timeline shows access checking benign inode A, attacker replacing path with a symlink, and open reaching protected inode B; bottom uses trusted dirfd and openat2 constraints to return a stable descriptor." width="98%"}

*Figure: original animated diagram based on the warnings in [`access(2)`](https://man7.org/linux/man-pages/man2/access.2.html), the Linux path-walk model in [`path_resolution(7)`](https://man7.org/linux/man-pages/man7/path_resolution.7.html), and resolution constraints in [`openat2(2)`](https://man7.org/linux/man-pages/man2/openat2.2.html).*

Safer designs avoid “check then reopen”:

- open relative to a trusted directory descriptor with `openat()` or `openat2()`;
- request `RESOLVE_BENEATH`, `RESOLVE_IN_ROOT`, `RESOLVE_NO_SYMLINKS`, `RESOLVE_NO_MAGICLINKS` or `RESOLVE_NO_XDEV` according to policy;
- use `O_NOFOLLOW`, `O_EXCL` and narrow access modes where their exact semantics match the operation;
- inspect the opened object with `fstat()` if type, owner or device remains a policy condition;
- perform all subsequent I/O through the descriptor rather than reconstructing its pathname; and
- for creation/publication, combine safe resolution with the durability protocol from Chapter 09.

`openat2()` resolution flags are policy tools, not a universal recipe. `RESOLVE_NO_SYMLINKS` can be unnecessarily strict for a trusted tree that intentionally uses symlinks. `RESOLVE_BENEATH` constrains escape above a directory but should be combined with mount and magic-link rules if those matter. State the intended namespace and choose the smallest rule that enforces it.

<details>
<summary><strong>Linux C example: create a file beneath a trusted directory without following symlinks</strong></summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <fcntl.h>
#include <linux/openat2.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/stat.h>
#include <sys/syscall.h>
#include <unistd.h>

static void die(const char *operation) {
    perror(operation);
    exit(EXIT_FAILURE);
}

static int open_new_beneath(int directory_fd, const char *relative_path) {
    struct open_how how = {
        .flags = O_WRONLY | O_CREAT | O_EXCL | O_CLOEXEC | O_NOFOLLOW,
        .mode = 0640,
        .resolve = RESOLVE_BENEATH |
                   RESOLVE_NO_SYMLINKS |
                   RESOLVE_NO_MAGICLINKS |
                   RESOLVE_NO_XDEV,
    };

    return (int)syscall(SYS_openat2, directory_fd, relative_path,
                        &how, sizeof how);
}

static void write_all(int fd, const char *data, size_t length) {
    size_t done = 0;
    while (done < length) {
        ssize_t written = write(fd, data + done, length - done);
        if (written > 0) {
            done += (size_t)written;
        } else if (written == -1 && errno == EINTR) {
            continue;
        } else {
            die("write");
        }
    }
}

int main(int argc, char **argv) {
    if (argc != 4) {
        fprintf(stderr, "usage: %s TRUSTED_DIRECTORY RELATIVE_NAME CONTENT\n",
                argv[0]);
        return EXIT_FAILURE;
    }

    int directory_fd = open(argv[1], O_RDONLY | O_DIRECTORY | O_CLOEXEC);
    if (directory_fd == -1) die("open trusted directory");

    int file_fd = open_new_beneath(directory_fd, argv[2]);
    if (file_fd == -1) {
        if (errno == ENOSYS) {
            fputs("openat2 is not supported by this kernel\n", stderr);
        }
        die("openat2 relative output");
    }

    // Validate the already-open object; never reopen argv[2] after this check.
    struct stat status;
    if (fstat(file_fd, &status) == -1) die("fstat output");
    if (!S_ISREG(status.st_mode)) {
        fputs("output is not a regular file\n", stderr);
        return EXIT_FAILURE;
    }

    write_all(file_fd, argv[3], strlen(argv[3]));
    if (fsync(file_fd) == -1) die("fsync output");
    if (close(file_fd) == -1) die("close output");

    // The new directory entry is a separate durable object (Chapter 09).
    if (fsync(directory_fd) == -1) die("fsync directory");
    if (close(directory_fd) == -1) die("close directory");
    return EXIT_SUCCESS;
}
```

Compile and run on a Linux system with recent kernel headers:

```bash
cc -std=c11 -Wall -Wextra -O2 safe_openat2.c -o safe_openat2
mkdir -p ./safe-output
./safe_openat2 ./safe-output result.txt 'bounded content'
```

The example deliberately uses `O_EXCL`, so it only creates a new name and cannot overwrite an existing object. Safe replacement requires a temporary file plus synchronization and same-directory `rename`, as shown in Chapter 09. Older kernels may return `ENOSYS`; a security-sensitive application must use a carefully designed fallback or fail closed, not silently call an unconstrained `open()`.

</details>

TOCTOU is not limited to paths. A PID can exit and its number be reused; use a PID descriptor when object identity must survive. A device can be unplugged between query and command; hold the driver's lifetime reference. Policy and credentials can change after a cached decision; key caches by policy generation and object identity. The general solution is to couple validation with a stable reference and a defined commit point.

### **Memory and Execution Defenses**

Address-space isolation from Chapters 06 and 07 prevents one ordinary process from directly addressing another's pages and prevents user mode from writing kernel pages. Security hardening adds constraints intended to turn common memory-corruption primitives into faults or make reliable exploitation harder.

![W^X, ASLR, guard pages, stack canaries and control-flow hardening remove different exploit assumptions.](assets/memory-execution-defenses.svg){fig-alt="Randomized process address space with read-execute code, read-write non-executable heap and stack, PROT_NONE guard page, and explanatory panels for W-X, ASLR and control-flow hardening." width="97%"}

*Figure: original explanatory diagram based on the Linux [kernel self-protection documentation](https://docs.kernel.org/security/self-protection.html), [`mmap(2)`](https://man7.org/linux/man-pages/man2/mmap.2.html), [`mprotect(2)`](https://man7.org/linux/man-pages/man2/mprotect.2.html), and [`randomize_va_space`](https://docs.kernel.org/admin-guide/sysctl/kernel.html#randomize-va-space).*

#### **W^X, Address-Space Randomization, and Guard Pages**

**W^X** (“write xor execute”) aims to prevent a memory page from being writable and executable simultaneously. Code and read-only data mappings should not be writable; heap and stack mappings should not be executable. Hardware NX/XD page-table bits and the kernel's VMA policy enforce the distinction.

W^X blocks a simple “inject bytes into writable memory and jump there” path. It does not prevent reuse of existing executable code, corruption of control data, logic bugs, or an information leak. JIT runtimes need controlled code generation: create writable non-executable memory, emit and validate code, transition it to read-execute, and avoid a long-lived RWX window. Production runtimes may use dual mappings or a dedicated compiler broker.

**Address Space Layout Randomization (ASLR)** varies locations of the executable (with PIE), shared libraries, mappings, heap and stack. If an attacker must guess $h$ independent random bits, a naive one-shot success probability is approximately:

$$
P(\text{correct guess}) = 2^{-h}.
$$

Real entropy is reduced by alignment, architecture, process layout, repeated attempts, shared mappings and leaked pointers. ASLR is therefore probabilistic defense in depth, not access control. A single reliable address disclosure can undermine much of it.

**Guard pages** are `PROT_NONE` gaps around stacks, allocators or sensitive regions. Crossing the boundary raises a page fault rather than immediately corrupting the next mapped object. They detect only accesses that reach the guard page; small in-object or same-page overflows can remain invisible.

Compiler and hardware defenses complement page policy:

- stack canaries detect selected overwrites before a function returns;
- control-flow integrity constrains indirect branch targets;
- shadow stacks protect return addresses in a separate structure;
- pointer authentication binds selected pointers to context on supporting architectures; and
- memory-safe languages remove broad classes of spatial and temporal errors before OS hardening is needed.

All have coverage assumptions. A canary does not prevent every data-only attack. CFI quality depends on the allowed target set. A guard page has page granularity. The correct conclusion is layered risk reduction, not “buffer overflows are solved.”

<details>
<summary><strong>Inspect executable mappings and hardening state on Linux</strong></summary>

```bash
# Address-space randomization policy: 0 off, 1 partial, 2 fuller randomization.
sysctl kernel.randomize_va_space

# Current shell mappings and their r/w/x permissions.
cat /proc/$$/maps

# ELF program headers: GNU_STACK should normally not request execute permission.
readelf -W -l /usr/bin/grep | grep -E 'LOAD|GNU_STACK|GNU_RELRO'

# ELF type DYN commonly indicates a PIE executable for ordinary programs.
readelf -W -h /usr/bin/grep | grep 'Type:'

# Compare selected mapping bases across fresh executions.
for i in 1 2 3; do
  sh -c 'grep -m1 "libc.*r-xp" /proc/$$/maps'
done
```

Exact results depend on distribution, architecture, linker and kernel configuration. A hardening flag describes one mitigation; it does not establish memory safety or prove that every loaded library follows the same policy.

</details>

### **Sandboxing and System-Call Filtering**

A **sandbox** runs code under a deliberately reduced authority set so compromise has fewer useful effects. A complete sandbox usually composes identity/privilege reduction, descriptor hygiene, object access policy, syscall filtering, namespace views, cgroup limits, memory defenses and audit. The setup order should become irreversible before untrusted input is parsed.

**seccomp** filters Linux syscall entry using a verified classic-BPF program over `struct seccomp_data`: architecture, syscall number, instruction pointer and raw argument values. A filter can allow, return an errno, trap, kill, log, trace or send a user notification. Filters are inherited across `fork()`/`clone()` and, when execution is allowed, across `execve()`; additional filters only make the effective result at least as restrictive.

![A seccomp BPF filter evaluates syscall metadata and selects allow, errno, log, trap, notification, or kill behavior.](assets/seccomp-policy-path-animated.svg){fig-alt="Animated syscall travels from sandboxed task to kernel seccomp BPF chain, then branches to ALLOW, LOG, ERRNO, TRAP/KILL, or USER_NOTIF; panels distinguish suitable and unsuitable policy questions." width="98%"}

*Figure: original animated diagram based on the official [seccomp BPF documentation](https://docs.kernel.org/userspace-api/seccomp_filter.html), [`seccomp(2)`](https://man7.org/linux/man-pages/man2/seccomp.2.html), and [`no_new_privs`](https://docs.kernel.org/userspace-api/no_new_privs.html).*

#### **seccomp and Policy Enforcement**

An unprivileged process normally sets `no_new_privs` before installing a filter. This prevents a malicious parent from applying a filter that changes a future privileged child in a way that yields more authority. Filters must check the syscall architecture as well as number because syscall-number tables differ by ABI.

seccomp intentionally cannot dereference user pointers. If it tried to inspect a pathname through a pointer, another thread could change the bytes before the kernel's actual syscall implementation copied them, recreating TOCTOU. seccomp is therefore strong at questions such as “may this task call `mount` at all?” and weak at “may it open this particular inode?” Object policy belongs in DAC/ACL, an LSM such as SELinux/AppArmor, Landlock, or a descriptor-based broker.

Policy construction has two broad styles:

| Style | Strength | Risk |
|---|---|---|
| Default deny / allowlist | Exposes only known required syscalls | Brittle across libc, architecture, optional code paths and updates |
| Default allow / denylist | Easier compatibility; removes selected dangerous calls | New or overlooked syscalls remain reachable |

A security sandbox generally aims toward an allowlist, but tracing one successful run is not enough. Error handling, locale, DNS, signals, threading, allocation failure, startup, shutdown and library updates can use different syscalls. Tests should deliberately exercise these paths and verify the policy on each supported ABI.

<details>
<summary><strong>Linux C example: demonstrate a small libseccomp restriction</strong></summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <seccomp.h>
#include <stdio.h>
#include <stdlib.h>
#include <sys/prctl.h>
#include <sys/stat.h>

static void die(const char *operation) {
    perror(operation);
    exit(EXIT_FAILURE);
}

static void check_seccomp(int result, const char *operation) {
    if (result < 0) {
        errno = -result;          // libseccomp returns negative errno values.
        die(operation);
    }
}

int main(void) {
    // This demonstration defaults to allow and denies directory creation.
    // A production sandbox should use a tested allowlist for its real workload.
    scmp_filter_ctx filter = seccomp_init(SCMP_ACT_ALLOW);
    if (filter == NULL) die("seccomp_init");

    check_seccomp(seccomp_rule_add(filter, SCMP_ACT_ERRNO(EPERM),
                                   SCMP_SYS(mkdir), 0),
                  "seccomp_rule_add mkdir");
    check_seccomp(seccomp_rule_add(filter, SCMP_ACT_ERRNO(EPERM),
                                   SCMP_SYS(mkdirat), 0),
                  "seccomp_rule_add mkdirat");

    // Once set, execve cannot add privilege through setuid/file capabilities.
    if (prctl(PR_SET_NO_NEW_PRIVS, 1, 0, 0, 0) == -1) {
        seccomp_release(filter);
        die("PR_SET_NO_NEW_PRIVS");
    }

    check_seccomp(seccomp_load(filter), "seccomp_load");
    seccomp_release(filter);      // Kernel retains the installed filter.

    errno = 0;
    if (mkdir("seccomp-should-fail", 0700) == -1 && errno == EPERM) {
        puts("mkdir blocked with EPERM as required");
        return EXIT_SUCCESS;
    }

    fputs("unexpected sandbox result\n", stderr);
    return EXIT_FAILURE;
}
```

Compile on a Linux system with libseccomp development headers:

```bash
cc -std=c11 -Wall -Wextra -O2 seccomp_demo.c -o seccomp_demo -lseccomp
./seccomp_demo
```

This is intentionally **not** a complete sandbox: default allow leaves nearly the entire kernel interface reachable. Its purpose is to show irreversible filter installation, explicit failure action and testable behavior. A real policy should pin supported architectures, define unknown-syscall behavior, minimize the allowlist, handle `clone`/`execve` deliberately, and compose object and resource controls.

</details>

Seccomp user notification allows a supervisor to receive selected syscall events and reply. It can support emulation or policy brokers, but it reintroduces distributed state and race questions. The supervisor must reason about target lifetime, descriptor injection, pointer data, interruption and who is allowed to control the listener. It should not be assumed to provide atomic pathname authorization automatically.

Linux Security Modules (LSMs) provide hooks at security-relevant object operations. SELinux can enforce centrally administered label/type policy; AppArmor emphasizes path-oriented profiles; Landlock lets even an unprivileged process add stackable restrictions to itself and descendants. Landlock governs kernel objects rather than merely syscall numbers, so it complements seccomp.

![A sandbox composes dropped privilege, namespaces, cgroups, LSM or Landlock object policy, seccomp, and memory defenses around a process that still shares the host kernel.](assets/sandbox-defense-in-depth.svg){fig-alt="Nested sandbox layers: untrusted no-new-privileges process, seccomp, LSM or Landlock, cgroup resource governance and namespaces, with W-X/ASLR inside and shared host kernel warning outside." width="95%"}

*Figure: original explanatory diagram based on the Linux [seccomp](https://docs.kernel.org/userspace-api/seccomp_filter.html), [Landlock](https://docs.kernel.org/userspace-api/landlock.html), [LSM](https://docs.kernel.org/userspace-api/lsm.html), [namespaces](https://man7.org/linux/man-pages/man7/namespaces.7.html), and [cgroup v2](https://docs.kernel.org/admin-guide/cgroup-v2.html) documentation.*

| Mechanism | Primary decision unit | Best question | Important limitation |
|---|---|---|---|
| seccomp | Syscall number and raw scalar arguments | Which kernel entry points may execute? | Not pathname/inode or information-flow policy |
| Landlock | Process domain and filesystem/network objects | Which object operations may this process access? | ABI/filesystem coverage must be checked; self-restriction only adds limits |
| SELinux/AppArmor-style LSM | System-wide labels/profiles and hooks | Which domains may operate on which object classes? | Policy complexity and deployment configuration are part of TCB |
| Namespace | Kernel-resource view | Which process IDs, mounts, networks or IDs are visible? | Not a fine-grained access-control policy by itself |
| cgroup | Hierarchical resource accounting/control | How much CPU, memory, PIDs or I/O may a workload consume? | Does not decide which files or syscalls are allowed |

Sandbox launchers should verify that each requested mechanism exists and was applied. Continuing after `seccomp_load()`, namespace setup or Landlock restriction fails can silently invert the intended policy. Observability should expose the actual effective policy generation and launch result, not merely the desired configuration file.

### **Namespaces and Resource Isolation**

A Linux **namespace** makes selected global-looking kernel resources appear as different instances to different processes. It changes the names and views through which a task reaches resources; it does not create another kernel. Processes in separate namespaces still issue syscalls to the same host kernel and can still share objects deliberately through inherited descriptors, shared mounts, network links or IPC.

The principal namespace types are:

| Namespace | Isolated view | Security-relevant detail |
|---|---|---|
| Mount | Mount list and filesystem hierarchy | Propagation and shared subtrees can carry mount events across boundaries |
| PID | Process-ID number space and process hierarchy | Inner PID 1 has lifecycle/signal responsibilities; outer namespaces can still see descendants |
| Network | Interfaces, addresses, routes, ports, firewall state, abstract Unix sockets | Connectivity requires explicit veth/bridge or other links |
| IPC | System V IPC and POSIX message queues | Does not isolate every descriptor-based IPC channel |
| UTS | Hostname and domain name | Mostly identity presentation, not an authorization boundary |
| User | UID/GID mappings and capability scope | “Root inside” can map to an unprivileged host UID |
| Cgroup | View of cgroup paths | Does not itself impose the resource limits |
| Time | Selected boot/monotonic clock offsets | Does not virtualize every clock or external timestamp |

![Processes in different namespace sets see distinct PID, mount, network, IPC, UTS and user-ID views while sharing the host kernel.](assets/namespace-isolated-views.svg){fig-alt="Two Linux namespace sets show different PID, mount, network, IPC, UTS and user views; inside UID 0 maps to host UID 100000 and has capabilities only in the child namespace scope." width="98%"}

*Figure: original explanatory diagram based on [`namespaces(7)`](https://man7.org/linux/man-pages/man7/namespaces.7.html), [`user_namespaces(7)`](https://man7.org/linux/man-pages/man7/user_namespaces.7.html), and the Linux kernel's [user-namespace resource-control guidance](https://docs.kernel.org/admin-guide/namespaces/resource-control.html).*

Namespace operations are represented by kernel objects. `/proc/PID/ns/*` entries act as handles; `clone()`/`unshare()` create or change selected views and `setns()` joins a namespace through such a handle when authorized. Holding a namespace descriptor can therefore be authority to re-enter a view later.

User namespaces require special care. A non-root host UID can map to UID 0 inside a child user namespace and hold a broad effective capability set **in that namespace**. Whether a capability authorizes an operation depends on the user namespace that owns or governs the target resource. Inner root may change a hostname in a UTS namespace owned by its user namespace yet be unable to reconfigure the host network namespace.

This scoped privilege enables rootless containers, but it also exposes additional kernel paths to unprivileged callers. Systems that enable untrusted user namespaces should combine them with current kernels, memory/PID limits, LSM policy and deliberate namespace-creation limits. A namespace is an isolation mechanism whose own management interface is part of the attack surface.

Mount namespaces need propagation policy. If a mount is shared with a peer group, changes can propagate between namespaces in ways that surprise a “private filesystem view” assumption. Container setup commonly makes the intended subtree private or slave before constructing the root. The visible root should also avoid dangerous host devices, writable kernel pseudo-filesystems and unintended host paths.

`chroot()` changes pathname resolution root but is not a general security boundary. It does not isolate PIDs, networks, credentials, syscalls or resources; a suitably privileged process can often escape or undermine it. A secure filesystem sandbox combines a controlled mount namespace/root, object policy, dropped privilege and descriptor hygiene.

<details>
<summary><strong>Inspect namespace identity and create a disposable view</strong></summary>

```bash
# Namespace symlink targets identify namespace objects for this process.
readlink /proc/self/ns/{user,mnt,pid,net,ipc,uts,cgroup,time} 2>/dev/null

# Compare another process when permission allows.
target_pid=1
ls -l /proc/$target_pid/ns 2>/dev/null

# Show UID/GID maps of the current user namespace.
cat /proc/self/uid_map
cat /proc/self/gid_map

# A disposable user+UTS namespace. Availability depends on host policy.
# -r maps the caller to root inside; the host identity remains unprivileged.
unshare --user --map-root-user --uts sh -c '
  echo "inside: uid=$(id -u), namespace=$(readlink /proc/self/ns/user)"
  hostname sandbox-name
  echo "hostname=$(hostname)"
'

# Observe mount/PID/network views without changing them.
findmnt
ps -eo pid,ppid,user,comm
ip link show 2>/dev/null
```

Do not interpret `uid=0` inside the example as host root. Inspect the UID map and namespace handles. Some systems disable unprivileged user namespaces or restrict `unshare`; a launcher requiring them must fail closed or use an explicitly documented alternative.

</details>

#### **cgroups and Resource Governance**

Namespaces answer “which instance can the workload see?” **Control groups (cgroups)** answer “how are limited resources accounted for and distributed among workloads?” cgroup v2 organizes processes in one hierarchy and attaches controllers for CPU, memory, PIDs, I/O, cpusets and other resources.

Resource control is part of security because availability can be attacked. A process that cannot read a secret may still deny service by forking indefinitely, allocating page cache and anonymous memory, consuming CPU, filling I/O queues or creating pressure that kills an unrelated process.

![cgroup v2 applies hierarchical limits, protections, weights and observations to child workloads.](assets/cgroup-v2-governance.svg){fig-alt="Root cgroup branches into interactive, batch and sandbox groups with cpu weight or quota, memory low high or max, pids max, io max and cpuset settings; lower boxes distinguish limits, protections and observation." width="98%"}

*Figure: original explanatory diagram based on the authoritative Linux kernel [cgroup v2 documentation](https://docs.kernel.org/admin-guide/cgroup-v2.html) and [`cgroups(7)`](https://man7.org/linux/man-pages/man7/cgroups.7.html).*

cgroup v2 distinguishes several policy forms:

- a **limit** is a maximum such as `memory.max`, `pids.max`, `cpu.max` or `io.max`;
- a **protection** reserves or favors usage under contention, such as `memory.low`;
- a **weight** distributes contested capacity proportionally, such as `cpu.weight`; and
- an **allocation** exclusively assigns a finite resource and cannot be overcommitted.

CPU bandwidth uses a quota and period. With `cpu.max = Q P`, the group can consume approximately $Q$ microseconds of CPU time per $P$ microseconds before throttling, across its eligible CPUs:

$$
\text{maximum average CPU capacity} \approx \frac{Q}{P}\ \text{CPU cores}.
$$

For example, `50000 100000` corresponds to roughly half of one CPU of average runtime. This is not latency isolation: runnable tasks can still experience scheduling delay, and bursts, parent limits, CPU affinity and other classes affect behavior.

Memory controls have intentionally different semantics:

| Interface | Meaning |
|---|---|
| `memory.current` | Accounted current usage |
| `memory.low` | Best-effort protection from reclaim under pressure |
| `memory.high` | Throttling/reclaim pressure boundary; normally does not immediately invoke OOM |
| `memory.max` | Hard usage limit; unresolved pressure can invoke OOM within the cgroup |
| `memory.events` | Counters for low/high/max/OOM-related events |

`pids.max` limits the number of tasks and directly mitigates fork bombs. Hitting it makes `fork()`/`clone()` fail, so applications and supervisors must handle failure without corrupting state. `io.max` can bound bytes or operations per second for selected devices. `cpuset.cpus` and memory-node controls constrain placement but must remain subsets of ancestor assignments.

Hierarchical inheritance is fundamental: a child cannot consume resources forbidden by an ancestor. Limits can be overcommitted across siblings because they are ceilings, while actual available capacity remains finite. Delegating a subtree requires controlling which files the delegate may write and preventing tasks from being moved across unauthorized boundaries.

Resource limits also have side effects that belong in API design:

- a memory limit may increase reclaim latency before OOM;
- a cgroup-local OOM kill can terminate one or several workload processes;
- CPU throttling can violate lock-holder or heartbeat timing assumptions;
- PID exhaustion can block helper creation needed for recovery; and
- I/O throttling can delay journal, audit or shutdown work if those services share the constrained group.

The policy should reserve enough capacity for graceful failure and place control-plane components outside the workload's failure domain where appropriate.

<details>
<summary><strong>Inspect cgroup v2 state and launch a bounded transient scope</strong></summary>

```bash
# Determine whether the unified cgroup v2 filesystem is mounted.
findmnt -t cgroup2

# Locate this shell in the hierarchy and inspect available controllers.
cat /proc/self/cgroup
cat /sys/fs/cgroup/cgroup.controllers

# Read effective usage and pressure where available.
cat /sys/fs/cgroup/memory.current 2>/dev/null
cat /sys/fs/cgroup/memory.events 2>/dev/null
cat /sys/fs/cgroup/cpu.stat 2>/dev/null
cat /proc/pressure/{cpu,memory,io} 2>/dev/null

# On a systemd host, create a transient user scope when policy permits.
# This example caps memory and task count and requests half one CPU average.
systemd-run --user --scope \
  -p MemoryMax=256M \
  -p TasksMax=64 \
  -p CPUQuota=50% \
  -- /usr/bin/grep kernel input.txt
```

The read-only commands are broadly safe; the transient launch depends on a functioning user service manager and delegated controllers. Verify the resulting cgroup and effective limits rather than assuming every requested property was accepted.

</details>

Namespaces and cgroups are complementary but orthogonal. A process can be in a private PID namespace with no memory limit, or tightly memory-limited while seeing the host process tree. Container runtimes compose both with capabilities, seccomp, mounts, LSM policy and image/runtime configuration; the next chapter analyzes that composition as a deployment abstraction.

### **Auditing, Logging, and Incident Evidence**

**Logging** records events for operational or application purposes. **Security auditing** records selected security-relevant decisions and state changes so they can support accountability, detection, policy verification and incident reconstruction. Audit is not prevention: a perfectly recorded unauthorized write is still an unauthorized write.

Useful evidence answers:

- **who** initiated the action: login identity, current effective identity, groups/domain and peer context;
- **what** operation was requested and with which non-secret parameters;
- **which object** was involved: stable inode/device, process, socket, label or policy object, plus a human-readable name where useful;
- **when and where** it happened: event sequence, trusted time source, host/boot/container identity;
- **what result** occurred: allow/deny, syscall return, partial effect and error; and
- **which policy/configuration generation** produced the decision.

Linux audit can mark a syscall event in kernel context and emit several records with one timestamp/serial, such as `SYSCALL`, `CWD`, `PATH`, `EXECVE` and `PROCTITLE`. Userspace `auditd` receives records over Netlink, writes/rotates local logs and can dispatch or forward events. Multiple records must be joined by event identity before interpretation.

![Security events pass through kernel audit selection, a bounded backlog, auditd, protected collection, and analysis.](assets/audit-evidence-pipeline.svg){fig-alt="Security event enters kernel audit rules and event grouping, passes through bounded Netlink backlog to auditd and protected collection; panels cover selection quality, evidence failure modes and operational monitoring." width="97%"}

*Figure: original explanatory diagram based on the Linux [audit kernel interfaces](https://docs.kernel.org/core-api/kernel-api.html#audit-interfaces), [`auditctl(8)`](https://man7.org/linux/man-pages/man8/auditctl.8.html), the [Red Hat audit record model](https://docs.redhat.com/en/documentation/red_hat_enterprise_linux/7/html/security_guide/sec-understanding_audit_log_files), and [NIST SP 800-92](https://csrc.nist.gov/pubs/sp/800/92/final).*

The audit login UID (often `auid` or `loginuid`) is valuable because it can preserve the originating login identity while effective UID changes through `sudo` or a setuid transition. Effective IDs still matter because they describe authority at the event. Record both when attribution requires both the actor and the active privilege.

Evidence quality has several failure modes:

| Failure | Consequence | Mitigation |
|---|---|---|
| Kernel backlog overflow | Events are lost before `auditd` receives them | Size/monitor backlog, define failure action, alert on lost counters |
| Excessive broad rules | Performance cost and analyst signal loss | Select security hypotheses and key operations; test event volume |
| Missing success or return status | Cannot distinguish attempt from effect | Record result, exit value and partial-effect semantics |
| Mutable local-only logs | Compromised host erases evidence | Restrict access and forward promptly to another trust domain |
| Clock/host ambiguity | Cross-system timeline cannot be reconstructed | Time synchronization, boot/host IDs and monotonic/event sequence |
| Secrets in arguments or content | Logs create a new disclosure channel | Minimize fields, redact before logging, protect retention and access |
| Identity without object stability | Path now names something else | Include inode/device, labels, handle/object IDs and event correlation |

Auditing every syscall is rarely the right policy. High-value events often include authentication and privilege transitions, security policy changes, audit configuration changes, access to selected assets, executable loading from sensitive locations, denied LSM operations, namespace/cgroup administration and changes to trusted binaries or keys.

Audit configuration itself belongs to the TCB. A process able to disable auditing, modify rules or delete collected events can shape the evidence. Remote collection helps, but the transport credentials, collector, retention policy and analyst access then enter the evidence trust chain.

<details>
<summary><strong>Inspect Linux audit state and correlate one event</strong></summary>

```bash
# Administrative status: enabled state, backlog, lost-event count and failure mode.
sudo auditctl -s

# List active rules and their searchable keys.
sudo auditctl -l

# Search recent syscall/file events and interpret numeric fields.
sudo ausearch --start recent -m SYSCALL,PATH,CWD -i

# Search events carrying a configured rule key.
sudo ausearch --start today -k protected-output -i

# SELinux/LSM denials where applicable.
sudo ausearch --start recent -m AVC,USER_AVC,SELINUX_ERR -i

# Service logs are complementary operational evidence, not kernel audit.
journalctl --since '10 minutes ago' --no-pager
```

Audit tools and privileges differ by distribution. When reading raw output, group records with the same `msg=audit(timestamp:serial)` identifier before drawing conclusions. Monitor `lost` and backlog fields: an empty search is not proof that no event occurred when the collection path was disabled or overflowing.

</details>

Audit should close a feedback loop. Test that expected allow and deny events are generated, shipped, retained, searchable and alertable. Measure queue loss and ingestion delay. During incident exercises, verify that evidence remains available after the workload or host has been compromised. A policy that exists only on paper is not an effective control.

### **Threat-Modeling the Running Pipeline**

Consider the complete command again:

```bash
cat input.txt | grep kernel > result.txt
```

A minimal security claim might be:

> Given an unprivileged attacker who controls file contents and scheduling but not the kernel, boot policy or administrator-owned parent directories, the pipeline reads only the selected input object, writes only the selected output object, executes approved binaries without gaining privilege, remains within a defined CPU/memory/PID/I/O budget, and emits attributable evidence for policy-relevant failures.

This statement is intentionally conditional. It names attacker power, trusted components, allowed effects and resource/evidence requirements. It does not claim safety against a compromised host kernel or malicious administrator.

![The shell pipeline can be threat-modeled as a chain of credentials, stable descriptors, process boundaries, resource budgets, and auditable effects.](assets/pipeline-security-threat-model.svg){fig-alt="Shell opens input, pipe and output and delegates descriptors to cat and grep; lower panels map symlink, descriptor leak, parser bug, resource exhaustion and audit-loss threats to specific controls." width="98%"}

*Figure: original end-to-end synthesis grounded in Linux [`openat2(2)`](https://man7.org/linux/man-pages/man2/openat2.2.html), [`pipe(7)`](https://man7.org/linux/man-pages/man7/pipe.7.html), [`execve(2)`](https://man7.org/linux/man-pages/man2/execve.2.html), [seccomp](https://docs.kernel.org/userspace-api/seccomp_filter.html), [cgroup v2](https://docs.kernel.org/admin-guide/cgroup-v2.html), and Linux audit sources cited above.*

The security-relevant execution sequence is:

1. **Establish the launch identity.** The shell has real/effective IDs, groups, capability sets, namespaces, cgroup membership, LSM context, umask and open descriptors. These define its initial protection domain.
2. **Resolve executable and data objects.** The shell searches parent directories and opens `input.txt` and the output under VFS DAC/ACL, capabilities, mount and LSM checks. A hardened launcher uses trusted directory descriptors and constrained resolution where path replacement is in the threat model.
3. **Create authority channels.** `pipe()` creates two descriptor endpoints. Input, output and pipe descriptors are object capabilities in practice: each child should receive only the endpoints required by its role.
4. **Construct child descriptor tables.** After `fork()`, the shell uses `dup2()` and closes every unused end. `O_CLOEXEC` prevents unrelated descriptors from crossing `execve()`. Keeping an extra pipe write end open can also block EOF indefinitely, turning descriptor hygiene into availability correctness.
5. **Perform credential/execution transitions.** The loader checks executable metadata and policy. `no_new_privs` can rule out privilege-gaining execution. A fixed executable or trusted lookup path avoids executing an attacker-selected program.
6. **Constrain compromised utilities.** `cat` needs reads from one descriptor and writes to another; `grep` needs pipe reads and output writes. A sandbox can remove pathname access, extra network/device visibility and unused syscalls after descriptors are installed.
7. **Enforce memory and resource boundaries.** W^X/ASLR reduce exploitation reliability. cgroup limits stop one pipeline from exhausting host memory, PIDs, CPU or I/O, while the launcher handles limit failures and child exits correctly.
8. **Record important decisions.** Audit can capture failed object opens, privilege/sandbox setup failures and protected-output changes with original login identity, active credentials, object identity, outcome and policy generation.

Threat-to-control mapping prevents feature-list security:

| Threat | Attack path | Required control and residual risk |
|---|---|---|
| Output symlink race | Replace `result.txt` between check and open | Resolve once beneath trusted dirfd; stable descriptor. Parent/mount policy remains trusted. |
| Descriptor leak | Child inherits key, socket or directory fd | `O_CLOEXEC`, explicit descriptor manifest, close unused fds. `/proc`/broker paths may still reacquire authority. |
| Executable substitution | Attacker changes `PATH` or binary | Fixed trusted executable/loader environment, immutable package policy. Trusted binary may still contain bugs. |
| Parser memory corruption | Crafted input compromises `grep` | Memory safety/hardening plus least authority, seccomp and object sandbox. Shared kernel remains exposed through allowed syscalls. |
| Confused deputy | Privileged shell/broker opens client-chosen path | Narrow typed request and rights-reduced handle delegation. Broker protocol/code remains TCB. |
| Fork/memory/CPU exhaustion | Repeated pipelines or pathological input | cgroup PID, memory, CPU and I/O policy plus admission control. Limits can still cause application-level failure. |
| Audit evasion | Flood queue or erase local records | Bounded selective rules, loss monitoring, protected off-host collection. Events before policy activation may be absent. |

The descriptor contract can be reviewed as a tiny access matrix:

| Process | Input file | Pipe read | Pipe write | Output file | Namespace lookup after exec |
|---|---:|---:|---:|---:|---:|
| shell | open/setup only | close | close | open/setup only | only launch/control needs |
| `cat` | read | none | write | none | ideally none |
| `grep` | none | read | none | write | only libraries/config explicitly required |

That table is more actionable than saying “children run as the same user.” It identifies exactly which handles must exist after `execve()` and makes leaks testable by inspecting `/proc/PID/fd` under appropriate permissions.

Not every ordinary interactive shell should impose this hardened policy. The point is to derive controls from the claim. A batch service processing untrusted uploads needs a stronger domain than a user directly running trusted local utilities. Security mechanisms should be proportional to asset value, attacker capability, operational cost and recovery requirements.

### **Comparison and Summary**

Protection mechanisms answer different questions and compose at different points:

| Mechanism | Core question | Decision object | Principal strength | Principal blind spot |
|---|---|---|---|---|
| Unix mode bits | Does owner/group/other have `rwx`? | Inode and path traversal | Compact, universal DAC | Coarse identity classes |
| POSIX ACL | Does this named user/group have effective rights? | One filesystem object | Fine-grained object-centric sharing | Inheritance/mask administration complexity |
| Object capability / descriptor | Does holder possess this object-right reference? | Stable kernel object | Explicit delegation and least authority | Revocation and ambient reacquisition |
| Linux capabilities | May task perform this privileged operation class? | Many kernel objects | Splits traditional root privilege | Some units are broad, especially `CAP_SYS_ADMIN` |
| LSM / Landlock | Does domain policy allow this object operation? | Security hooks and object context | MAC or stackable object policy | Coverage and policy configuration are TCB |
| `no_new_privs` | May future exec add authority? | Task credential transition | Irreversible privilege-gain barrier | Does not remove current IDs, caps or descriptors |
| seccomp | May this syscall shape enter the kernel? | Syscall metadata | Reduces reachable kernel surface | Not full object/information-flow policy |
| Namespace | Which resource instance is visible? | Kernel namespace object | Efficient view isolation | Shared kernel; no resource budget by itself |
| cgroup | How much resource may this hierarchy use? | Process group and controller | Availability containment/accounting | No file/syscall authorization |
| W^X/ASLR/guards | How exploitable is memory corruption? | Virtual-memory mappings/control data | Removes common exploit assumptions | Does not repair logic or memory-safety bugs |
| Audit | What decision/effect can be reconstructed? | Event and evidence pipeline | Accountability, detection and diagnosis | Does not prevent the recorded operation |

The most important conceptual distinctions are:

- **identity is not authority**: two same-UID tasks can hold different descriptors, capabilities and sandboxes;
- **naming is not authorization**: a pathname identifies a candidate object, while policy decides operations on the resolved object;
- **visibility is not permission**: namespaces hide or rename resources, but object and syscall policy still matter;
- **resource control is not access control**: cgroups protect availability, not confidentiality of a file;
- **hardening is not memory safety**: W^X and ASLR change exploitability rather than eliminating the bug;
- **logging is not prevention**: audit supports evidence only if selection, delivery, integrity and loss monitoring work; and
- **one control is not a sandbox**: meaningful confinement is the verified composition of privilege, handles, object policy, syscalls, views and budgets.

For each security-sensitive operation, ask:

1. **Claim:** What exact confidentiality, integrity, availability and accountability property is required?
2. **Adversary:** Which inputs, processes, credentials, races and resources can the attacker control?
3. **Subject:** Which immutable credential and namespace context is used for the decision?
4. **Object:** Is authorization bound to a stable object/handle or to a mutable name checked earlier?
5. **Operation:** Are rights specific, minimal and attenuated before delegation?
6. **Transition:** Can `fork()`/`execve()`, setuid metadata, capabilities or inherited descriptors expand authority?
7. **Mediation:** Does every effective path reach an equivalent reference check, including caches and alternate APIs?
8. **Containment:** What remains possible after code execution inside the least-trusted component?
9. **Availability:** Which CPU, memory, PID, I/O, queue and storage budgets limit abuse and preserve recovery?
10. **Evidence:** Can allow/deny outcomes, effective identity, stable object and policy generation be reconstructed, and is loss detectable?
11. **TCB:** Which hardware, kernel, policy, broker, identity and collection components must be correct?
12. **Failure:** Does unsupported or failed security setup stop safely rather than silently widening authority?

The chapter's central path is:

$$
\text{threat model}
\rightarrow \text{credentialed subject}
\rightarrow \text{stable object and operation}
\rightarrow \text{reference decision}
\rightarrow \text{least-authority execution domain}
\rightarrow \text{resource containment}
\rightarrow \text{auditable effect}.
$$

Protection is strongest when authority is explicit, narrow, short-lived and carried by stable handles; isolation reduces the interactions a compromised component can exploit; security is the end-to-end argument that these controls remain effective under the declared attacker and failure model. The next chapter builds on these mechanisms to compare virtual machines, containers and kernel architectures: each moves the isolation boundary, changes the TCB, and pays a different cost for compatibility, performance and assurance.
